# TP3 — Analisis de sentimiento en tweets

**Diplomatura en Inteligencia Artificial — Marcelo Vieira**

Esta es la notebook principal del TP3: resume el problema, los datos, las decisiones
y los resultados. El desarrollo completo esta en las 6 notebooks de
[`notebooks/`](./notebooks/), enlazadas al final.

---

## 1. El problema

Dado el texto de un tweet, predecir si expresa un sentimiento **negativo o positivo**.

- Tarea de NLP: clasificacion supervisada de texto.
- Dataset: **Sentiment140** — 1.600.000 tweets en ingles etiquetados automaticamente
  (por emoticones), mas 498 tweets etiquetados **a mano** como test.
- Requisito mandatorio de la consigna: los resultados finales se calculan sobre el
  **dataset completo**, sin muestras.

## 2. Calidad de las etiquetas

![Distribucion de clases](imgs/02_distribucion_clases.png)

El dataset se etiqueto automaticamente segun si el tweet tenia un emoticon (`:)` o
`:(`), que despues fue borrado del texto. Eso genera errores concretos, por ejemplo
"Hiccups. Just what I need before retiring to my reading room" quedo etiquetado como
negativo, cuando el texto (sin el emoticon original) no transmite eso con claridad.

Como evidencia dura, se buscaron textos identicos repetidos con etiquetas opuestas:
**2.225 textos** (0,43% del dataset). Es un **piso**, no el total del ruido real: solo
detecta duplicados exactos letra por letra. Por eso esos registros se conservan (ademas
de que la consigna exige usar todos los datos): eliminarlos daria una falsa sensacion
de haber limpiado el problema.

## 3. Que muestra el EDA

![Longitud de tweets](imgs/02_longitud_tweets.png)

**La longitud no discrimina** (misma distribucion en ambas clases). La senial esta en
el **vocabulario**:

![Top palabras por clase](imgs/02_top_palabras.png)

Los **bigramas** muestran una diferencia interesante entre clases: los positivos son
mas consistentemente positivos (`good morning`, `happy birthday`, `looking forward`),
mientras que los negativos mezclan casos claros (`sorry hear`, `dont want`) con
bigramas ambiguos que no aportan mucho por si solos (`dont know`, `feel like`).

![Top bigramas por clase](imgs/02_top_bigramas.png)

Los signos de puntuacion no definen sentimiento por si solos (solo dan enfasis), pero
la exclamacion aparece bastante mas en positivos que en negativos (35,4% vs 24,8%),
mientras que la interrogacion aparece casi igual en las dos clases (10,8% vs 10,2%).

## 4. Decisiones tomadas

Cada decision se tomo despues de ver un ejemplo concreto del problema que resolvia:

| Decision | Motivo |
| --- | --- |
| Modelo **binario** (negativo/positivo) | El training no tiene ejemplos neutrales; no se puede aprender una clase sin datos |
| Conservar **negaciones** (`not`, `don't`, `no`) | Sin ellas no se distingue `good` de `not good` — confirmado despues en los coeficientes del modelo |
| Apostrofes: pegar (`don't` -> `dont`), no expandir | El vectorizador corta por el apostrofe y pierde la negacion si no se pega |
| Menciones `@usuario` -> `usuariomencionado` | Borrar rompe la estructura de la oracion en tweets donde la mencion es sujeto |
| URLs -> `linkweb` | Mismo criterio que las menciones |
| Hashtags: sacar el `#`, conservar la palabra | Es contenido real del tweet |
| **TF-IDF (1-2 gramas) + Logistic Regression** | Eficiente para 1,6 M de tweets cortos e interpretable (coeficientes por n-grama) |
| Split 90/10 **estratificado con shuffle** | El archivo viene ordenado por clase |
| Duplicados conflictivos: **se conservan** | La consigna exige datos completos; se documentan como limitacion |
| Modelo final **reentrenado con el 100%** (1.600.000) | Requisito mandatorio, verificado con `assert` |

## 5. Resultados

| Evaluacion | accuracy | precision | recall | F1 |
| --- | ---: | ---: | ---: | ---: |
| Train del modelo de desarrollo (1.440.000) | 0,8466 | 0,8399 | 0,8564 | 0,8481 |
| Validacion (160.000 tweets no vistos) | 0,8249 | 0,8188 | 0,8343 | 0,8265 |

**Por que el accuracy solo no alcanza**: un modelo que siempre predijera "positivo"
sin leer el tweet, en este dataset balanceado 50/50, acertaria 50% (acierta toda una
clase y falla toda la otra por completo). El modelo entrenado saca 82%, muy por
encima de ese piso de comparacion.

![Matriz de confusion validacion](imgs/05_confusion_validacion.png)

De 160.000 tweets de validacion: 132.011 acertados, 27.989 errados. Los dos tipos de
error (14.697 negativos-predichos-positivos, 13.292 positivos-predichos-negativos)
son parecidos entre si, sin sesgo fuerte hacia un lado — consistente con que
precision y recall tambien salen parecidos entre las dos clases.

**Chequeo de overfitting**: la brecha entre train y validacion es de ~2,2 puntos de
accuracy -> no hay sobreajuste relevante.

## 6. Demo en vivo

El modelo final (guardado en `models/`) prediciendo tweets nuevos:

In [1]:
import sys
sys.path.insert(0, "notebooks")

import joblib
import pandas as pd
from utils import limpiar_tweets, VECTORIZER_JOBLIB, MODELO_JOBLIB

vec = joblib.load(VECTORIZER_JOBLIB)
lr = joblib.load(MODELO_JOBLIB)

ejemplos = [
    "I love this song, best concert ever!",
    "my phone died again and I lost all my photos",
    "cant wait to see you tomorrow!!",
    "not bad at all, actually pretty good",
    "I miss my dog so much... this house feels empty",
    "Great. Another monday. Yay...",          # sarcasmo: deberia fallar
]

proba = lr.predict_proba(vec.transform(limpiar_tweets(pd.Series(ejemplos))))[:, 1]
for texto, p in zip(ejemplos, proba):
    veredicto = "POSITIVO" if p >= 0.5 else "NEGATIVO"
    print(f"  p(pos)={p:.3f} -> {veredicto:8s} | {texto}")

  p(pos)=0.987 -> POSITIVO | I love this song, best concert ever!
  p(pos)=0.003 -> NEGATIVO | my phone died again and I lost all my photos
  p(pos)=0.971 -> POSITIVO | cant wait to see you tomorrow!!
  p(pos)=0.993 -> POSITIVO | not bad at all, actually pretty good
  p(pos)=0.004 -> NEGATIVO | I miss my dog so much... this house feels empty
  p(pos)=0.838 -> POSITIVO | Great. Another monday. Yay...


El ultimo ejemplo es **sarcasmo**: palabras literalmente positivas (`great`, `yay`)
en un mensaje negativo. Es una limitacion estructural de un modelo de bolsa de
palabras: no puede detectar que el sentido se invierte por el tono.

## 7. Interpretacion: que aprendio el modelo

![Coeficientes mas predictivos](imgs/06_coeficientes.png)

Entre los n-gramas que mas empujan a **positivo** aparecen varios con negacion:
`cant wait`, `not bad`, `no problem`, `no need`, `dont need`. Esto confirma la
decision de conservar las negaciones tomada antes de entrenar: si se hubiera sacado
el `not` de `not bad`, el bigrama hubiera quedado solo como `bad`, que el modelo
aprenderia como negativo — el sentido contrario al real.

### Similitud coseno (metrica de clase obligatoria)

Mide que tan parecidos son dos tweets segun el vocabulario que comparten (0 = nada en
comun, 1 = mismo vocabulario). Se aplico de dos formas:

- **Vecinos mas cercanos**: un tweet con "freakin cooool i love twitter" encontro
  vecinos por compartir la palabra "freakin" y otros por compartir la frase
  "i love twitter".
- **Busqueda de etiquetas mal puestas**: sobre una muestra de 300 tweets negativos, se
  encontro 1 caso con similitud maxima (coseno=1,000) y etiqueta opuesta, pero resulto
  ser el mismo tipo de caso que los duplicados exactos ya conocidos, no un hallazgo
  nuevo. Limitacion: 300 tweets es una muestra chica frente al total.

## 8. Limitaciones

1. **Ruido de etiquetado automatico**: al menos 0,43% del dataset son textos
   identicos con etiquetas opuestas (piso, no total); pone un techo a cualquier
   modelo entrenado sobre este dataset.
2. **Clase neutral ausente en training**: el modelo principal es binario porque no
   hay ejemplos de esa clase para entrenar con ella.
3. **Sarcasmo e ironia**: son estructuralmente invisibles para un modelo de bolsa de
   palabras (ver el ejemplo de la demo en vivo).
4. **Alcance recortado**: por tiempo, esta entrega no incluye modelado de topicos,
   analisis sistematico de sarcasmo ni analisis por usuario; se priorizo profundizar
   el analisis obligatorio (calidad de datos, preprocesamiento, evaluacion,
   similitud coseno) en vez de cubrir mas superficie con menos detalle.

## 9. Conclusiones

1. TF-IDF (1-2 gramas) + Logistic Regression, entrenado con los **1.600.000 tweets
   completos**, clasifica polaridad con ~82% de accuracy, muy por encima del piso de
   50% de un modelo que no aprende nada.
2. La senial de sentimiento es **lexica** y las **negaciones importan**: se verifico
   con ejemplos concretos en el preprocesamiento y se confirmo despues con los
   coeficientes del modelo entrenado.
3. La **similitud coseno** sirvio para encontrar tweets con vocabulario parecido y
   para buscar (sin encontrar casos nuevos en esta muestra) etiquetas mal puestas mas
   alla de los duplicados exactos.
4. **Mejora futura**: profundizar el analisis de errores (sarcasmo, ironia) y el
   modelado de topicos con el mismo nivel de detalle que el resto del trabajo, y
   sumar datos neutrales reales para plantear el problema de 3 clases.

---

## Notebooks de detalle

| Notebook | Contenido |
| --- | --- |
| [01_carga_y_validacion](./notebooks/01_carga_y_validacion.ipynb) | Carga completa, clases, calidad de las etiquetas |
| [02_eda](./notebooks/02_eda.ipynb) | Longitud, vocabulario, bigramas y signos de puntuacion por clase |
| [03_preprocesamiento](./notebooks/03_preprocesamiento.ipynb) | Limpieza documentada y persistencia |
| [04_entrenamiento](./notebooks/04_entrenamiento.ipynb) | TF-IDF + LR, split, chequeo de overfitting y modelo final con el 100% |
| [05_evaluacion](./notebooks/05_evaluacion.ipynb) | Baseline, metricas, matriz de confusion |
| [06_interpretacion](./notebooks/06_interpretacion.ipynb) | Coeficientes y similitud coseno |